***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 4 章：可见度空间](4_0_introduction.ipynb)
    * 上一节：[4.3 二元干涉仪](4_3_the_2-element_interferometer.ipynb)
    * 下一节：[4.5.1 $uv$ 轨迹跟踪](4_5_1_uv_coverage_uv_tracks.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from scipy.special import j1
except Exception:
    j1 = None
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 4.4 可见度函数<a id='visibility:sec:visibility_function'></a>

第 4.3 节得到的是一条基线、一个时刻、一个频率上的复相关输出。本节把这个单个输出推广为函数：当投影基线在 `uvw` 空间中变化时，相关器输出也随之变化，所有这些值共同构成*可见度函数*。干涉阵观测得到的是这个函数的一组离散样本。

可见度函数是连接测量和成像的桥梁。天空亮度 $I_\nu(l,m)$ 描述图像平面中的角结构；可见度 $V(u,v,w)$ 描述同一结构在空间频率平面中的投影。短基线靠近 `uv` 原点，主要测大尺度结构；长基线远离原点，主要测小尺度结构；可见度的相位则记录结构相对于相位中心的位置。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

# Figure 4.4.1: one Fourier kernel in the image plane.
N = 300
l = np.linspace(-0.5, 0.5, N)
m = np.linspace(-0.5, 0.5, N)
L, M = np.meshgrid(l, m)
u0, v0 = 7.0, 4.0
kernel = np.cos(2*np.pi*(u0*L + v0*M))
fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(kernel, extent=[l.min(), l.max(), m.min(), m.max()], origin='lower', cmap='RdBu_r', vmin=-1, vmax=1)
ax.arrow(0, 0, 0.20, 0.115, width=0.006, head_width=0.035, color='black', length_includes_head=True)
ax.text(0.23, 0.13, r'$(u,v)$', fontsize=13)
ax.set_xlabel('l')
ax.set_ylabel('m')
ax.set_title('Fringe pattern for one visibility sample')
fig.colorbar(im, ax=ax, shrink=0.82, label='real Fourier kernel')
fig.savefig(fig_dir / 'visibility_kernel_fringe.png', dpi=180, bbox_inches='tight')
plt.close(fig)

# Figure 4.4.2: source size and position in visibility amplitude/phase.
u = np.linspace(-35, 35, 1400)
sigmas = [0.018, 0.035, 0.070]
fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.5), constrained_layout=True)
for sigma in sigmas:
    amp = np.exp(-2*np.pi**2*sigma**2*u**2)
    axes[0].plot(u, amp, lw=1.8, label=fr'$\sigma={sigma:.3f}$ rad')
axes[0].set_xlabel('u')
axes[0].set_ylabel('normalized amplitude')
axes[0].set_title('Broader sources fade faster on long baselines')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False, fontsize=9)

l0_values = [0.00, 0.025, 0.055]
for l0 in l0_values:
    vis = np.exp(-2j*np.pi*u*l0)
    phase = np.unwrap(np.angle(vis))
    axes[1].plot(u, phase, lw=1.8, label=fr'$l_0={l0:.3f}$ rad')
axes[1].set_xlabel('u')
axes[1].set_ylabel('unwrapped phase [rad]')
axes[1].set_title('Source offset appears as phase slope')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False, fontsize=9)
fig.savefig(fig_dir / 'visibility_size_position.png', dpi=180, bbox_inches='tight')
plt.close(fig)

# Figure 4.4.3: uniform disk visibility nulls.
rho = np.linspace(0.001, 40, 1200)
R = 0.06
x = 2*np.pi*rho*R
if j1 is not None:
    disk_vis = 2*j1(x)/x
else:
    # fallback series/numpy approximation is unnecessary for normal use; keep a smooth illustrative envelope.
    disk_vis = np.sinc(x/np.pi) * np.cos(0.35*x)
first_null = 3.831705970 / (2*np.pi*R)
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ax.plot(rho, disk_vis, color='#2f6f9f', lw=2.0)
ax.axhline(0, color='0.75', lw=0.8)
ax.axvline(first_null, color='#c53030', ls='--', lw=1.8, label='first null')
ax.text(first_null+0.8, 0.58, r'$\rho_1\simeq0.61/R$', color='#c53030', fontsize=12)
ax.set_xlabel(r'baseline radius $\rho=\sqrt{u^2+v^2}$')
ax.set_ylabel('normalized visibility')
ax.set_title('Uniform disk: angular size from the first visibility null')
ax.set_xlim(0, 40)
ax.set_ylim(-0.25, 1.05)
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.savefig(fig_dir / 'visibility_disk_nulls.png', dpi=180, bbox_inches='tight')
plt.close(fig)


### 4.4.1 从相关输出到函数<a id='visibility:sec:from_sample_to_function'></a>

第 4.3 节得到的相位跟踪后相关输出可以写成

$$
V(u,v,w)=\iint \frac{A(l,m)I_\nu(l,m)}{n}
\exp\left[-2\pi i\left(ul+vm+w(n-1)\right)\right] dl\,dm.
$$

这里 $A(l,m)$ 是主波束或方向响应，$I_\nu(l,m)$ 是天空比强度，$(l,m,n)$ 是相对于相位中心的方向余弦，$(u,v,w)$ 是以波长为单位的投影基线。因子 $1/n$ 来自球面立体角元到局部平面坐标的变换：

$$
d\Omega = \frac{dl\,dm}{n}.
$$

这条公式说明，可见度不是一个抽象数学量，而是天空亮度乘以仪器响应后，再与一个复指数条纹相乘并积分的结果。每一个 $(u,v,w)$ 点都对应一组不同的条纹，因此也对应天空的一个不同空间频率投影。


### 4.4.2 傅里叶近似<a id='visibility:sec:fourier_visibility'></a>

若视场足够小，$l$ 和 $m$ 都很小，则

$$
n=\sqrt{1-l^2-m^2}\approx 1-\frac{l^2+m^2}{2}.
$$

进一步若 $w(n-1)$ 在相位中可以忽略，并且主波束在视场内变化缓慢，便得到熟悉的二维傅里叶关系：

$$
V(u,v)\approx \iint I_\nu(l,m)e^{-2\pi i(ul+vm)}dl\,dm.
$$

反过来，若 $V(u,v)$ 被完整采样，则可以通过逆变换恢复天空亮度：

$$
I_\nu(l,m)\approx \iint V(u,v)e^{2\pi i(ul+vm)}du\,dv.
$$

实际干涉阵只能在离散的 `uv` 点上采样，因此真正得到的是被采样函数截断后的可见度。后续成像中的脏图像、脏波束、加权和 CLEAN，都来自这个“不完整傅里叶采样”的事实。


![单个可见度样本对应的条纹](figures/visibility_kernel_fringe.png)

**图 4.4.1** 单个 $(u,v)$ 点对应图像平面中的一组条纹。可见度可以理解为天空亮度与这组复条纹相乘后的积分。


### 4.4.3 可见度幅度：角尺度的信息<a id='visibility:sec:visibility_amplitude'></a>

可见度幅度最直接地反映源结构在对应空间频率上的强弱。中心点源的傅里叶变换为常数，因此所有基线都能测到同样的幅度；扩展源的傅里叶变换会随基线长度增加而下降，因为长基线对应更细的条纹，亮暗条纹在扩展源上相互抵消。

以位于相位中心的一维高斯源为例：

$$
I(l)=I_0\exp\left(-\frac{l^2}{2\sigma^2}\right).
$$

它的可见度仍是高斯：

$$
V(u)=I_0\sqrt{2\pi}\sigma\exp\left(-2\pi^2\sigma^2u^2\right).
$$

因此图像中越宽的源，在 `uv` 平面中下降越快。这个尺度互易关系是干涉测量中判断“源是否被分辨”的基础：当长基线上的幅度显著低于短基线，说明源具有可解析的角大小。


### 4.4.4 可见度相位：位置信息<a id='visibility:sec:visibility_phase'></a>

相位对源位置非常敏感。若一个点源位于 $l=l_0$，则

$$
I(l)=I_0\delta(l-l_0),
$$

其可见度为

$$
V(u)=I_0e^{-2\pi iul_0}.
$$

幅度仍为常数 $I_0$，但相位随 $u$ 线性变化：

$$
\phi(u)=-2\pi ul_0.
$$

二维情形中，位于 $(l_0,m_0)$ 的点源对应

$$
V(u,v)=I_0e^{-2\pi i(ul_0+vm_0)}.
$$

这就是相位天体测量的基本原因。源位置的微小偏移会在长基线上积累成可测相位变化；反过来，仪器相位误差也会伪装成源位置或结构误差，因此校准在干涉测量中至关重要。


![源大小与位置在可见度中的表现](figures/visibility_size_position.png)

**图 4.4.2** 左图显示高斯源越宽，可见度幅度越快随基线长度下降。右图显示源位置偏离相位中心时，可见度相位随 $u$ 出现线性斜率。


### 4.4.5 圆盘源与可见度零点<a id='visibility:sec:disk_visibility'></a>

许多经典角直径测量并不依赖完整成像，而是直接利用可见度幅度随基线长度的变化。对均匀圆盘源，若角半径为 $R$，其归一化可见度只依赖 $\rho=\sqrt{u^2+v^2}$：

$$
\frac{V(\rho)}{V(0)}=\frac{2J_1(2\pi \rho R)}{2\pi \rho R},
$$

其中 $J_1$ 是一阶贝塞尔函数。第一个零点出现在

$$
2\pi\rho R \approx 3.8317,
$$

因此

$$
\rho_1\approx \frac{0.61}{R}.
$$

这给出了一个非常直接的解释：基线越长，可探测的角尺度越小；当可见度首次降到零时，条纹亮暗区域恰好把圆盘源的贡献抵消掉。早期恒星角直径测量和现代长基线干涉中的许多模型拟合，都可以看成这一思想的推广。


![均匀圆盘的可见度零点](figures/visibility_disk_nulls.png)

**图 4.4.3** 均匀圆盘源的归一化可见度。第一个零点给出角半径的直接尺度信息。


### 4.4.6 采样、噪声与实际测量<a id='visibility:sec:sampled_visibility'></a>

真实观测不会得到连续的 $V(u,v,w)$，而是得到一组带噪声的样本。若用 $S(u,v,w)$ 表示采样函数，用 $N$ 表示噪声，则可写成

$$
V_{\rm obs}=S\,V+N.
$$

采样函数由阵列布局、观测时间、频率覆盖、数据标记和权重共同决定。第 4.5 节将讨论 `uv` 覆盖如何产生；第 5 章会进一步说明，当不完整采样的可见度被反变换回图像平面时，得到的并不是真天空，而是真天空与脏波束的卷积。

因此，可见度函数有两层意义。作为连续函数，它是天空亮度的傅里叶表示；作为观测数据，它是一组受仪器和采样限制的复数。射电干涉成像的全部工作，就是在这两层意义之间谨慎往返。


### 4.4.7 本节要点

可见度函数把天空亮度写到空间频率域中。在小视场、窄带、远场且 $w$ 项可忽略的条件下，可见度近似等于天空亮度的二维傅里叶变换。可见度幅度主要反映角尺度和结构是否被分辨；可见度相位主要记录相对于相位中心的位置。真实干涉阵只采样这个函数的一部分，因此后续必须讨论 `uv` 覆盖、采样函数、脏波束和成像反演。


***

* 下一节：[4.5.1 $uv$ 轨迹跟踪](4_5_1_uv_coverage_uv_tracks.ipynb)
